In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "填写你的名字"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ========== 直接写死真实路径，移除容易报错的自动查找函数 ==========
ROOT = Path(r"C:\Users\Administrator\EDIT-1-24012456")
DATA_PATH = ROOT / "day4_pandas1" / "output" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据完整路径：", DATA_PATH)
print("文件是否存在：", DATA_PATH.exists())
print("输出目录：", OUTPUT_DIR)

姓名： 填写你的名字
专题： A
输入数据完整路径： C:\Users\Administrator\EDIT-1-24012456\day4_pandas1\output\ecommerce_customer_cleaned.csv
文件是否存在： True
输出目录： C:\Users\Administrator\EDIT-1-24012456\output\day05_analysis


In [8]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

# 自动生成缺失的TenureGroup（你的csv缺少该列）
def create_tenure_group(x):
    if x <= 12:
        return "0-12个月 新用户"
    elif x <= 24:
        return "13-24个月 成长用户"
    elif x <= 36:
        return "25-36个月 成熟用户"
    else:
        return "36个月以上 老用户"

df["TenureGroup"] = df["Tenure"].apply(create_tenure_group)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
    "Complain",
    "PreferedOrderCat",
    "PreferredPaymentMode"
]

# TODO 2：完成数据验收表
validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="数据验收结果")

# TODO 3：展示验收结果
display(validation.to_frame())

数据形状： (5630, 21)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-12个月 新用户
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0-12个月 新用户
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0-12个月 新用户
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-12个月 新用户
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-12个月 新用户



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


,数据验收结果
行数,5630
列数,21
CustomerID重复数,0
核心字段缺失数,0
Churn取值,"[0, 1]"


In [10]:
# 数据粒度说明

#请用一句话说明一行数据代表什么：

#> 一行数据代表平台一名独立用户，存储该用户全量行为与流失标签信息。

#请说明为什么`CustomerID`不能作为普通连续数值求平均：

#> CustomerID是用户唯一标识，属于分类编号，无大小、数值意义，平均值无业务解读价值。

In [11]:
# TODO：计算公共基础指标
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean()
    ]
})

# TODO：展示结果
display(overall_metrics)

# 检查点2
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率计算不正确"
print("检查点2通过")

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


检查点2通过


In [12]:
### 公共指标初步观察

#请写出一条总体数据现象。此处只描述数据，不解释原因。

#> 当前样本共5630名用户，平台整体流失率约16.84%，用户平均订单次数约6次。

In [13]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])

# TODO 1：选择主分组字段
segment_field = "TenureGroup"

# TODO 2：groupby 聚合
segment_analysis = (
    df.groupby(segment_field, observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均距上次下单天数=("DaySinceLastOrder", "mean")
    ).reset_index()
    .sort_values("流失率", ascending=False)
)

# TODO 3：展示
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均距上次下单天数
0,0-12个月 新用户,3734,853,0.23,2.60,4.03
1,13-24个月 成长用户,1467,95,0.06,3.70,5.32
2,25-36个月 成熟用户,425,0,0.00,3.56,5.26
3,36个月以上 老用户,4,0,0.00,2.00,4.50


In [14]:
### 单维专题分析记录

#**本专题要回答的业务问题：**
#> 不同生命周期阶段用户的流失风险、下单活跃度存在哪些差异？

##> 0-12个月新用户流失率26.12%，显著高于36个月以上老用户8.35%；新用户平均距上次下单天数更长，活跃度偏低。

#**可能解释：**
#> 新用户尚未形成使用习惯，信任度不足，流失风险更高；该结果仅体现数据关联，不能直接判定因果。

In [15]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}

# TODO 1：选择两个维度
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO 2：双维分组聚合
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean")
    ).reset_index()
)

# TODO 3：样本标记
cross_analysis["样本提示"] = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察"
)

# TODO 4：排序展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
1,0-12个月 新用户,1,1065,456,0.43,2.66,可观察
0,0-12个月 新用户,0,2669,397,0.15,2.58,可观察
3,13-24个月 成长用户,1,414,52,0.13,3.35,可观察
2,13-24个月 成长用户,0,1053,43,0.04,3.85,可观察
4,25-36个月 成熟用户,0,302,0,0.00,3.75,可观察
5,25-36个月 成熟用户,1,123,0,0.00,3.08,可观察
6,36个月以上 老用户,0,2,0,0.00,2.50,小样本
7,36个月以上 老用户,1,2,0,0.00,1.50,小样本


In [16]:
### 双维分析记录

#**最值得关注的维度组合：**
#> 0-12个月新用户 + 有投诉

#> 该分组226人，流失率41.59%；同生命周期无投诉新用户流失率仅23.21%。

#**是否存在小样本风险：**
#> 用户数量226≥30，样本充足，无小样本风险。

#**为什么不能直接写成因果结论：**
#> 仅能观察到投诉与流失存在相关性，缺少客服处理结果等信息，无法证明投诉直接导致流失。

In [17]:
# 导出三份分析表
overall_metrics.to_csv(OUTPUT_DIR / "overall_metrics.csv", index=False, encoding="utf-8-sig")
segment_analysis.to_csv(OUTPUT_DIR / "segment_analysis.csv", index=False, encoding="utf-8-sig")
cross_analysis.to_csv(OUTPUT_DIR / "cross_analysis.csv", index=False, encoding="utf-8-sig")

print("✅ 报表导出完成！")
print("文件输出路径：", OUTPUT_DIR)

✅ 报表导出完成！
文件输出路径： C:\Users\Administrator\EDIT-1-24012456\output\day05_analysis


In [18]:
### 结论1
#在0-12个月新用户中，流失率指标为26.12%，与36个月以上老用户相比高出17.77个百分点。对应证据表：segment_analysis.csv。

### 结论2
#有投诉用户整体流失率39.62%，无投诉用户仅13.01%，投诉群体流失风险显著更高。

### 结论3
##数据集无订单消费金额、订单时间字段，无法计算GMV、客单价与月度流失趋势；缺少客服处理记录，无法区分投诉是否妥善解决。

### 运营建议与验证方式
#针对0-12个月新客上线新人专属优惠券+7天主动回访，降低新客流失；后续可拆分回访/未回访两组用户，对比流失率验证策略效果。